In [4]:
# ==============================================================================
# ورشة هيئة الشارقة للآثار: الذكاء الاصطناعي في توثيق المواقع والقطع الأثرية
# المحور الثالث: كشف الشذوذ الأثري بالاستشعار عن بعد والرؤية الحاسوبية (مُصحح)
# ==============================================================================

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider
import ipywidgets as widgets

print("🚀 جاري تهيئة بيئة التحليل الأثري الطيفي...")

# 1. توليد مشهد طيفي يُحاكي منطقة أثرية صحراوية مع تحويله تلقائياً إلى صيغة uint8 القياسية
def generate_synthetic_archaeological_scene():
    np.random.seed(42)
    size = 400

    # خلفية رملية صحراوية متغيرة الكثافة
    base_sand = np.random.normal(0.65, 0.05, (size, size))
    sand_dunes = np.sin(np.linspace(0, 10, size))[:, None] * 0.08
    soil_background = np.clip(base_sand + sand_dunes, 0.2, 0.9)

    # النطاقات الطيفية الأساسية
    red_band = soil_background.copy()
    green_band = soil_background * 0.85
    blue_band = soil_background * 0.70
    nir_band = soil_background * 0.90 # الأشعة تحت الحمراء القريبة

    # مَعْلم 1: أساسات سور مستطيل مدفون
    rr, cc = np.meshgrid(np.arange(size), np.arange(size))
    wall_mask = ((rr > 80) & (rr < 220) & ((cc == 80) | (cc == 220))) | \
                ((cc > 80) & (cc < 220) & ((rr == 80) | (rr == 220)))
    wall_mask = cv2.dilate(wall_mask.astype(np.uint8), np.ones((5, 5), np.uint8)).astype(bool)

    red_band[wall_mask] += 0.18
    nir_band[wall_mask] -= 0.12

    # مَعْلم 2: مدافن ركامية دائرية (Burial Cairns / Tumuli)
    cairn_centers = [(150, 320), (300, 120), (320, 290)]
    for cy, cx in cairn_centers:
        dist_sq = (rr - cy)**2 + (cc - cx)**2
        cairn_mask = dist_sq < 14**2
        red_band[cairn_mask] += 0.22
        nir_band[cairn_mask] -= 0.15

    # مَعْلم 3: مسار قناة فلج قديم جاف (Paleochannel)
    falaj_curve = (np.sin(np.linspace(0, 3, size)) * 50 + 260).astype(int)
    for r in range(size):
        c = falaj_curve[r]
        if 0 <= c < size:
            falaj_mask = (abs(cc - c) < 3) & (rr == r)
            nir_band[falaj_mask] += 0.15
            red_band[falaj_mask] -= 0.08

    # تحويل الصورة إلى uint8 (نطاق 0-255) لحل متطلب cv2.putText و OpenCV 8-bit
    rgb = (np.stack([np.clip(red_band, 0, 1),
                     np.clip(green_band, 0, 1),
                     np.clip(blue_band, 0, 1)], axis=-1) * 255).astype(np.uint8)

    return rgb, red_band, nir_band

rgb_scene, b_red, b_nir = generate_synthetic_archaeological_scene()

# 2. خوارزمية استخراج وتحليل الشذوذ الأثري بالرؤية الحاسوبية
def detect_archaeological_anomalies(sensitivity=0.18, min_area=30, blur_kernel=5):
    # حساب مؤشر التباين الأثري المخصص (ANDI)
    andi = (b_red - b_nir) / (b_red + b_nir + 1e-6)
    andi_norm = cv2.normalize(andi, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # تنقية الضوضاء الرملية
    k = blur_kernel if blur_kernel % 2 == 1 else blur_kernel + 1
    blurred = cv2.GaussianBlur(andi_norm, (k, k), 0)

    # العتبة التكيفية لاستخلاص الشذوذ
    thresh_val = int(255 * (1.0 - sensitivity))
    _, thresh = cv2.threshold(blurred, thresh_val, 255, cv2.THRESH_BINARY)

    # استخراج المعالم الهندسية
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detection_overlay = rgb_scene.copy()
    detected_count = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_area:
            detected_count += 1
            x, y, w, h = cv2.boundingRect(cnt)
            # تمييز الشذوذ المكتشف باللون الأخضر الصريح (0, 255, 0)
            cv2.rectangle(detection_overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(detection_overlay, f"Site #{detected_count}", (x, max(14, y - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, lineType=cv2.LINE_AA)

    # عرض اللوحات المقارنة
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=110)

    axes[0].imshow(rgb_scene)
    axes[0].set_title("1. المشهد الفضائي الطبيعي (RGB)\nصعوبة رصد المعالم بالعين المجردة", fontsize=11, fontweight='bold')
    axes[0].axis("off")

    heatmap = axes[1].imshow(andi_norm, cmap="inferno")
    axes[1].set_title("2. مؤشر الشذوذ الطيفي المخصص (ANDI)\nإبراز التغير في كثافة التربة والرطوبة", fontsize=11, fontweight='bold')
    axes[1].axis("off")
    plt.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(detection_overlay)
    axes[2].set_title(f"3. الاكتشاف الآلي بالذكاء الاصطناعي\nتم رصد: {detected_count} موقع/هيكل محتمل", fontsize=11, fontweight='bold', color='darkgreen')
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# 3. تشغيل الواجهة التفاعلية
print("✅ اكتمل الإعداد بنجاح. حرّك المؤشرات أدناه لاستعراض الكشف الآلي:")
interact(
    detect_archaeological_anomalies,
    sensitivity=FloatSlider(min=0.05, max=0.40, step=0.02, value=0.18, description="حساسية الكشف:"),
    min_area=IntSlider(min=10, max=200, step=10, value=30, description="الحد الأدنى للحجم:"),
    blur_kernel=IntSlider(min=1, max=15, step=2, value=5, description="تصفية الضوضاء:")
);

🚀 جاري تهيئة بيئة التحليل الأثري الطيفي...
✅ اكتمل الإعداد بنجاح. حرّك المؤشرات أدناه لاستعراض الكشف الآلي:


interactive(children=(FloatSlider(value=0.18, description='حساسية الكشف:', max=0.4, min=0.05, step=0.02), IntS…

In [5]:
!pip install arabic-reshaper python-bidi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 3.5 MB/s eta 0:00:00


In [7]:
import arabic_reshaper
from bidi.algorithm import get_display

def get_arabic_text(text):
    # إعادة تشكيل الحروف العربية لتكون متصلة بشكل صحيح
    reshaped_text = arabic_reshaper.reshape(text)
    # تصحيح اتجاه الكتابة من اليمين إلى اليسار
    bidi_text = get_display(reshaped_text)
    return bidi_text

# تعديل خوارزمية استخراج وتحليل الشذوذ الأثري لعرض النصوص العربية بشكل صحيح
def detect_archaeological_anomalies_fixed(sensitivity=0.18, min_area=30, blur_kernel=5):
    # حساب مؤشر التباين الأثري المخصص (ANDI)
    andi = (b_red - b_nir) / (b_red + b_nir + 1e-6)
    andi_norm = cv2.normalize(andi, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # تنقية الضوضاء الرملية
    k = blur_kernel if blur_kernel % 2 == 1 else blur_kernel + 1
    blurred = cv2.GaussianBlur(andi_norm, (k, k), 0)

    # العتبة التكيفية لاستخلاص الشذوذ
    thresh_val = int(255 * (1.0 - sensitivity))
    _, thresh = cv2.threshold(blurred, thresh_val, 255, cv2.THRESH_BINARY)

    # استخراج المعالم الهندسية
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    detection_overlay = rgb_scene.copy()
    detected_count = 0

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > min_area:
            detected_count += 1
            x, y, w, h = cv2.boundingRect(cnt)
            # تمييز الشذوذ المكتشف باللون الأخضر الصريح
            cv2.rectangle(detection_overlay, (x, y), (x + w, y + h), (0, 255, 0), 2)
            cv2.putText(detection_overlay, f"Site #{detected_count}", (x, max(14, y - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, lineType=cv2.LINE_AA)

    # عرض اللوحات المقارنة مع تصحيح النصوص العربية
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=110)

    axes[0].imshow(rgb_scene)
    title_1 = get_arabic_text("1. المشهد الفضائي الطبيعي (RGB)\nصعوبة رصد المعالم بالعين المجردة")
    axes[0].set_title(title_1, fontsize=11, fontweight='bold')
    axes[0].axis("off")

    heatmap = axes[1].imshow(andi_norm, cmap="inferno")
    title_2 = get_arabic_text("2. مؤشر الشذوذ الطيفي المخصص (ANDI)\nإبراز التغير في كثافة التربة والرطوبة")
    axes[1].set_title(title_2, fontsize=11, fontweight='bold')
    axes[1].axis("off")
    plt.colorbar(heatmap, ax=axes[1], fraction=0.046, pad=0.04)

    axes[2].imshow(detection_overlay)
    title_3 = get_arabic_text(f"3. الاكتشاف الآلي بالذكاء الاصطناعي\nتم رصد: {detected_count} موقع/هيكل محتمل")
    axes[2].set_title(title_3, fontsize=11, fontweight='bold', color='darkgreen')
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# تشغيل الواجهة التفاعلية المعدلة
print("✅ اكتمل الإعداد بنجاح. حرّك المؤشرات أدناه لاستعراض الكشف الآلي:")
interact(
    detect_archaeological_anomalies_fixed,
    sensitivity=FloatSlider(min=0.05, max=0.40, step=0.02, value=0.18, description="حساسية الكشف:"),
    min_area=IntSlider(min=10, max=200, step=10, value=30, description="الحد الأدنى للحجم:"),
    blur_kernel=IntSlider(min=1, max=15, step=2, value=5, description="تصفية الضوضاء:")
);

✅ اكتمل الإعداد بنجاح. حرّك المؤشرات أدناه لاستعراض الكشف الآلي:


interactive(children=(FloatSlider(value=0.18, description='حساسية الكشف:', max=0.4, min=0.05, step=0.02), IntS…

In [2]:
!pip install python-pptx -q

import os
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from google.colab import files

print("🏛️ جاري بناء حزم العروض التقديمية الاحترافية لهيئة الشارقة للآثار...")

# لوحة الألوان السيادية
COLOR_NAVY_DARK = RGBColor(15, 23, 42)     # #0F172A خلفية داكنة فخمة
COLOR_SLATE_BOX = RGBColor(30, 41, 59)     # #1E293B بطاقات داكنة
COLOR_SAND_GOLD = RGBColor(212, 163, 115)  # #D4A373 ذهبي رملي للتراث
COLOR_TERRACOTTA = RGBColor(178, 58, 34)   # #B23A22 قرميدي آثاري
COLOR_WHITE = RGBColor(255, 255, 255)
COLOR_MUTED = RGBColor(148, 163, 184)      # نصوص فرعية رمادية
COLOR_LIGHT_BG = RGBColor(248, 250, 252)   # خلفية بيضاء نقية للشرائح الفاتحة

MODULES = [
    {
        "id": "1",
        "title": "مقدمة في الذكاء الاصطناعي وتطبيقاته الأثرية",
        "filename": "العرض_1_مقدمة_الذكاء_الاصطناعي_الآثار.pptx",
        "accent": COLOR_TERRACOTTA,
        "badge": "المحور الأول: المفاهيم والتأسيس الأكاديمي",
        "s2_title": "التحول من الأرشفة الساكنة إلى التحليل الدلالي",
        "s2_left_head": "الرقمنة الساكنة التقليدية",
        "s2_left_body": "• جداول Excel وقواعد بيانات معزولة وغير مترابطة.\n• صور فوتوغرافية وأوراق مسح تفتقر للمعلومات الحاسوبية الدلالية.\n• استرجاع وتصنيف يدوي يستنزف أكثر من 70% من وقت الباحث الأثري.",
        "s2_right_head": "التوثيق الذكي التنبؤي (2026)",
        "s2_right_body": "• تعرف آلي على الأنماط واستخراج السمات الزخرفية للقطع.\n• نماذج متعددة الوسائط (Multimodal AI) للفهرسة الفورية من الصور.\n• تطبيق معايير FAIR الدولية للربط بين مكتشفات متاحف ومواقع الشارقة.",
        "s3_steps": [
            ("التقاط الصورة/النقش", "تصوير الكسرة أو المسكوكة عبر كاميرا عالية الدقة"),
            ("معالجة الرؤية (Vision AI)", "عزل التآكل واستخراج معالم الخطوط والكتابات"),
            ("المطابقة بنموذج Ithaca", "استكمال الكلمات المتآكلة وتحديد عصر وتاريخ القطعة"),
            ("التصدير القياسي", "إنتاج بطاقة تعريفية آلية وفق معيار Dublin Core")
        ],
        "s4_kpis": [
            ("71%", "دقة نموذج Ithaca في ترميم واستكمال النقوش التاريخية التالفة"),
            ("84%", "دقة تحديد الموطن الجغرافي الأصلي للنصوص والمسكوكات"),
            ("80%", "تخفيض في الوقت المستهلك لإعداد بطاقات التوثيق المتحفية"),
            ("0.1mm", "دقة مطابقة قوالب ضرب المسكوكات الإسلامية المكتشفة بالشارقة")
        ],
        "notes": "التركيز على مسكوكات موقع المدام ودراهم مليحة لإثبات الجدوى الميدانية لأدوات الذكاء الاصطناعي أمام الإدارة."
    },
    {
        "id": "2",
        "title": "توثيق المواقع والتسجيل الميداني 3D",
        "filename": "العرض_2_توثيق_المواقع_والتسجيل_الميداني.pptx",
        "accent": COLOR_SAND_GOLD,
        "badge": "المحور الثاني: التوثيق والنمذجة الميدانية",
        "s2_title": "طفرة النمذجة ثلاثية الأبعاد: من SfM إلى 3DGS",
        "s2_left_head": "التصوير المساحي الكلاسيكي (SfM)",
        "s2_left_body": "• يتطلب مئات الصور المتداخلة وزمن معالجة طويل جداً.\n• يعاني أمام التباينات الحادة لضوء الشمس في صحراء الشارقة.\n• حاجة ماسة لمحطات عمل حاسوبية فائقة التعقيد بالموقع.",
        "s2_right_head": "رذاذ غاوس (3D Gaussian Splatting)",
        "s2_right_body": "• طفرة النمذجة: تمثيل المشهد كاملاً بسرعة معالجة فورية.\n• تصفح تفاعلي سلس للمواقع والمربعات بمعدل 60 إطاراً في الثانية.\n• العمل مباشرة من الهاتف والدرون دون الحاجة لأجهزة عملاقة.",
        "s3_steps": [
            ("المسح بالهاتف (LiDAR)", "استخدام Polycam لمسح المربع الأثري أو اللقية"),
            ("التسجيل الصوتي الحقلي", "تحويل إملاء الباحث الأثري الميداني إلى سجل حفر رقمي"),
            ("مقاطع الفخار الآلية", "استخراج Rim Profiles تلقائياً دون رسم يدوي مجهد"),
            ("المزامنة مع QGIS", "ربط السحابة النقطية والتوأم الرقمي بقواعد بيانات الهيئة")
        ],
        "s4_kpis": [
            ("60 FPS", "سرعة التصفح السلس للتوائم الرقمية بمواقع التنقيب"),
            ("75%", "توفير في زمن استخراج ورسم مقاطع حواف الأواني الفخارية"),
            ("1mm", "دقة قياس الأبعاد الواقعية عبر مستشعرات الليدار المحمولة"),
            ("100%", "حفظ رقمي دائم للسياق الطبقي للموقع قبل إزالة الطبقات")
        ],
        "notes": "التأكيد على أن الحفرية الأثرية بطبيعتها عملية تدميرية متحكم بها؛ ما يُحفر لا يمكن إعادته، لذا فإن التوأم الرقمي يحفظ الموقع للأبد."
    },
    {
        "id": "3",
        "title": "الكشف والتنبؤ بالآثار المدفونة والاستشعار عن بعد",
        "filename": "العرض_3_الكشف_والتنبؤ_والاستشعار_عن_بعد.pptx",
        "accent": RGBColor(6, 182, 212),
        "badge": "المحور الثالث: الاستشعار عن بعد والذكاء المكاني",
        "s2_title": "اختراق الرمال الصحراوية عبر رادار الفضاء (SAR)",
        "s2_left_head": "التصوير الضوئي الفضائي المحدود",
        "s2_left_body": "• يلتقط فقط المعالم السطحية الظاهرة للعين البشرية المجردة.\n• تغطية الرمال الصحراوية الزاحفة تحجب بالكامل الآثار المدفونة.\n• صعوبة تتبع قنوات المياه القديمة والأسوار المطمورة تحت السطح.",
        "s2_right_head": "الرادار الفضائي والليدار (SAR & LiDAR)",
        "s2_right_body": "• موجات الرادار الميكروية (L-band) تخترق الرمال الجافة لعمق 1-3 أمتار.\n• الارتداد التفاضلي يكشف كثافة الأساسات الحجرية المطمورة.\n• الليدار الجوي يعزل الكثبان الرملية والنباتات لإنتاج نماذج DTM عارية.",
        "s3_steps": [
            ("استدعاء صور Sentinel-1/2", "تحميل النطاقات الطيفية ورادار الفتحة الاصطناعية للمنطقة"),
            ("حساب مؤشر ANDI الأثري", "مقارنة النطاق الأحمر بنطاق الأشعة تحت الحمراء القريبة"),
            ("التصنيف بنموذج YOLO", "اكتشاف الأنماط الدائرية للمدافن والمستطيلة للأسوار"),
            ("توليد الإحداثيات الجغرافية", "تصدير خريطة اشتباه أثري عالية الاحتمالية لفرق المسح")
        ],
        "s4_kpis": [
            ("1 - 3m", "عمق اختراق موجات رادار SAR للرمال الصحراوية الجافة بالشارقة"),
            ("1000s", "كيلومترات مربعة تُفحص وتُحلل آلياً عبر الذكاء الاصطناعي في دقائق"),
            ("92%", "دقة النماذج التنبؤية في تمييز مدافن العصر البرونزي وقنوات الأفلاج"),
            ("Zero", "حفريات عشوائية؛ توجيه فرق المسح مباشرة لنقاط مؤكدة بنسب احتمالية")
        ],
        "notes": "استعراض كود Colab التفاعلي وشرح كيف تبرز الأساسات الأثرية في خريطة التباين الطيفي بالألوان الفسفورية."
    },
    {
        "id": "4",
        "title": "مراقبة حالة المواقع والقطع الأثرية ورصد التدهور",
        "filename": "العرض_4_مراقبة_حالة_المواقع_ورصد_التدهور.pptx",
        "accent": RGBColor(16, 185, 129),
        "badge": "المحور الرابع: الصيانة التنبؤية وإدارة المخاطر",
        "s2_title": "المراقبة رباعية الأبعاد (4D Time-Lapse) وحماية التراث",
        "s2_left_head": "الترميم العلاجي الكلاسيكي (رد الفعل)",
        "s2_left_body": "• التدخل فقط بعد حدوث التصدع الكبير أو انهيار جزء من الجدار.\n• تكاليف مالية باهظة وصعوبة بالغة في استعادة الحالة الأصلية.\n• غياب القياس الدقيق لمعدلات التآكل البطيئة الناتجة عن الرياح والأمطار.",
        "s2_right_head": "الصيانة التنبؤية الذكية (الاستباق)",
        "s2_right_body": "• مقارنة السحب النقطية (خوارزمية M3C2) لرصد الإزاحات المليمترية مبكراً.\n• التعرف الآلي على الشروخ ومعدل اتساعها عبر الرؤية الحاسوبية.\n• الكشف المبكر عن مرض البرونز والصدأ بالقطع المعدنية في المستودعات.",
        "s3_steps": [
            ("المسح الدوري المقارن", "إجراء مسح ليزري أو تصويري سنوي لنفس الموقع أو المبنى"),
            ("المطابقة السحابية (M3C2)", "مقارنة هندسية فائقة الدقة لعزل الفروق الناتجة عن التآكل"),
            ("التجزئة الدلالية للشروخ", "تصنيف عمق واتساع الشقوق وتحديد درجة خطورتها آلياً"),
            ("تقرير الاستجابة المؤتمت", "إصدار تنبيه عاجل لفرق الصيانة وفق مصفوفة المخاطر المعتمدة")
        ],
        "s4_kpis": [
            ("0.5mm", "أصغر إزاحة هيكلية أو تآكل يمكن للنظام رصده وتنبيه الإدارة به"),
            ("65%", "تخفيض في كلفة أعمال الترميم بفضل التدخل الوقائي الاستباقي"),
            ("100%", "أتمتة تقارير الحالة الإنشائية وتصنيف المخاطر وفق معايير EAMENA"),
            ("24/7", "استجابة ذكية لحماية القلاع والمواقع الأثرية المفتوحة من عوامل الطقس")
        ],
        "notes": "الختام باستعراض خارطة طريق التحول الرقمي وتأسيس وحدة الرصد الذكي داخل هيئة الشارقة للآثار."
    }
]

def create_slide_header(slide, prs, badge_text, slide_title, accent_color):
    """رسم رأس الشريحة الاحترافي بهوية حكومة الشارقة"""
    # شريط خلفية الرأس
    header_box = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, Inches(13.333), Inches(1.15))
    header_box.fill.solid()
    header_box.fill.fore_color.rgb = COLOR_NAVY_DARK
    header_box.line.fill.background()

    # خط سفلي فاخر بلون المحور
    line = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, Inches(1.15), Inches(13.333), Inches(0.06))
    line.fill.solid()
    line.fill.fore_color.rgb = accent_color
    line.line.fill.background()

    # نصوص الرأس
    tf = header_box.text_frame
    tf.word_wrap = True
    tf.margin_right = Inches(0.8)
    tf.margin_top = Inches(0.15)

    p0 = tf.paragraphs[0]
    p0.text = f"حكومة الشارقة — هيئة الشارقة للآثار | {badge_text}"
    p0.font.size = Pt(11)
    p0.font.bold = True
    p0.font.color.rgb = COLOR_SAND_GOLD
    p0.alignment = PP_ALIGN.RIGHT

    p1 = tf.add_paragraph()
    p1.text = slide_title
    p1.font.size = Pt(20)
    p1.font.bold = True
    p1.font.color.rgb = COLOR_WHITE
    p1.alignment = PP_ALIGN.RIGHT

def generate_professional_pptx(mod):
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]

    accent = mod["accent"]

    # =========================================================================
    # الشريحة 1: شريحة العنوان الملكية (Dark Hero Master Slide)
    # =========================================================================
    s1 = prs.slides.add_slide(blank_layout)
    bg1 = s1.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, Inches(13.333), Inches(7.5))
    bg1.fill.solid()
    bg1.fill.fore_color.rgb = COLOR_NAVY_DARK
    bg1.line.fill.background()

    # إطار زخرفي جانبي بلون المحور
    side_bar = s1.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(13.1), 0, Inches(0.233), Inches(7.5))
    side_bar.fill.solid()
    side_bar.fill.fore_color.rgb = accent
    side_bar.line.fill.background()

    # لوحة العنوان المركزية الفاخرة
    hero_card = s1.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(1.2), Inches(1.3), Inches(10.9), Inches(4.9))
    hero_card.fill.solid()
    hero_card.fill.fore_color.rgb = COLOR_SLATE_BOX
    hero_card.line.color.rgb = COLOR_SAND_GOLD
    hero_card.line.width = Pt(1.5)

    htf = hero_card.text_frame
    htf.word_wrap = True
    htf.margin_right = Inches(0.8)
    htf.margin_top = Inches(0.6)
    htf.margin_left = Inches(0.8)

    hp0 = htf.paragraphs[0]
    hp0.text = "حكومة الشارقة — هيئة الشارقة للآثار | المذكرة الرسمية SAA-CCS/1147/2026"
    hp0.font.size = Pt(14)
    hp0.font.bold = True
    hp0.font.color.rgb = COLOR_SAND_GOLD
    hp0.alignment = PP_ALIGN.RIGHT

    hp1 = htf.add_paragraph()
    hp1.text = mod["title"]
    hp1.font.size = Pt(30)
    hp1.font.bold = True
    hp1.font.color.rgb = COLOR_WHITE
    hp1.alignment = PP_ALIGN.RIGHT

    hp2 = htf.add_paragraph()
    hp2.text = f"\nورشة عمل: الذكاء الاصطناعي في توثيق المواقع والقطع الأثرية — {mod['badge']}"
    hp2.font.size = Pt(16)
    hp2.font.color.rgb = COLOR_MUTED
    hp2.alignment = PP_ALIGN.RIGHT

    # =========================================================================
    # الشريحة 2: شريحة المقارنة الثنائية المتقدمة (Split Screen Layout)
    # =========================================================================
    s2 = prs.slides.add_slide(blank_layout)
    bg2 = s2.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, Inches(13.333), Inches(7.5))
    bg2.fill.solid()
    bg2.fill.fore_color.rgb = COLOR_LIGHT_BG
    bg2.line.fill.background()
    create_slide_header(s2, prs, mod["badge"], mod["s2_title"], accent)

    # الصندوق الأيمن (الأسلوب القديم)
    box_r = s2.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(6.8), Inches(1.6), Inches(5.7), Inches(5.2))
    box_r.fill.solid()
    box_r.fill.fore_color.rgb = COLOR_WHITE
    box_r.line.color.rgb = RGBColor(226, 232, 240)
    box_r.line.width = Pt(1.5)
    rtf = box_r.text_frame
    rtf.word_wrap = True
    rtf.margin_right = Inches(0.4)
    rtf.margin_left = Inches(0.4)
    rtf.margin_top = Inches(0.4)
    rp0 = rtf.paragraphs[0]
    rp0.text = f"❌ {mod['s2_left_head']}"
    rp0.font.size = Pt(18)
    rp0.font.bold = True
    rp0.font.color.rgb = COLOR_TERRACOTTA
    rp0.alignment = PP_ALIGN.RIGHT
    rp1 = rtf.add_paragraph()
    rp1.text = f"\n{mod['s2_left_body']}"
    rp1.font.size = Pt(14)
    rp1.font.color.rgb = COLOR_NAVY_DARK
    rp1.alignment = PP_ALIGN.RIGHT

    # الصندوق الأيسر (الحل الذكي المتقدم)
    box_l = s2.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.8), Inches(1.6), Inches(5.7), Inches(5.2))
    box_l.fill.solid()
    box_l.fill.fore_color.rgb = COLOR_WHITE
    box_l.line.color.rgb = accent
    box_l.line.width = Pt(2)
    ltf = box_l.text_frame
    ltf.word_wrap = True
    ltf.margin_right = Inches(0.4)
    ltf.margin_left = Inches(0.4)
    ltf.margin_top = Inches(0.4)
    lp0 = ltf.paragraphs[0]
    lp0.text = f"✨ {mod['s2_right_head']}"
    lp0.font.size = Pt(18)
    lp0.font.bold = True
    lp0.font.color.rgb = accent
    lp0.alignment = PP_ALIGN.RIGHT
    lp1 = ltf.add_paragraph()
    lp1.text = f"\n{mod['s2_right_body']}"
    lp1.font.size = Pt(14)
    lp1.font.color.rgb = COLOR_NAVY_DARK
    lp1.alignment = PP_ALIGN.RIGHT

    # =========================================================================
    # الشريحة 3: شريحة مسار تدفق العمليات (Pipeline Process Flow)
    # =========================================================================
    s3 = prs.slides.add_slide(blank_layout)
    bg3 = s3.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, Inches(13.333), Inches(7.5))
    bg3.fill.solid()
    bg3.fill.fore_color.rgb = COLOR_LIGHT_BG
    bg3.line.fill.background()
    create_slide_header(s3, prs, mod["badge"], "خارطة الإجراءات والتدفق الحقلي والمكتبي (Field-to-Lab Pipeline)", accent)

    step_w = Inches(2.7)
    step_h = Inches(4.8)
    # 4 خطوات موزعة بالتساوي من اليمين إلى اليسار
    x_positions = [Inches(9.8), Inches(6.8), Inches(3.8), Inches(0.8)]

    for idx, (st_title, st_desc) in enumerate(mod["s3_steps"]):
        box_st = s3.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, x_positions[idx], Inches(1.7), step_w, step_h)
        box_st.fill.solid()
        box_st.fill.fore_color.rgb = COLOR_WHITE
        box_st.line.color.rgb = RGBColor(203, 213, 225)
        box_st.line.width = Pt(1.5)

        stf = box_st.text_frame
        stf.word_wrap = True
        stf.margin_right = Inches(0.25)
        stf.margin_left = Inches(0.25)
        stf.margin_top = Inches(0.4)

        sp0 = stf.paragraphs[0]
        sp0.text = f"الخطوة {idx+1}"
        sp0.font.size = Pt(14)
        sp0.font.bold = True
        sp0.font.color.rgb = accent
        sp0.alignment = PP_ALIGN.CENTER

        sp1 = stf.add_paragraph()
        sp1.text = st_title
        sp1.font.size = Pt(16)
        sp1.font.bold = True
        sp1.font.color.rgb = COLOR_NAVY_DARK
        sp1.alignment = PP_ALIGN.CENTER

        sp2 = stf.add_paragraph()
        sp2.text = f"\n{st_desc}"
        sp2.font.size = Pt(12)
        sp2.font.color.rgb = RGBColor(71, 85, 105)
        sp2.alignment = PP_ALIGN.RIGHT

    # =========================================================================
    # الشريحة 4: شريحة الإحصائيات والمؤشرات القيادية (Executive KPI Grid)
    # =========================================================================
    s4 = prs.slides.add_slide(blank_layout)
    bg4 = s4.shapes.add_shape(MSO_SHAPE.RECTANGLE, 0, 0, Inches(13.333), Inches(7.5))
    bg4.fill.solid()
    bg4.fill.fore_color.rgb = COLOR_NAVY_DARK
    bg4.line.fill.background()
    create_slide_header(s4, prs, mod["badge"], "الأثر التشغيلي والمؤشرات الرقمية الميدانية (Key Impact Metrics)", accent)

    kpi_w = Inches(5.6)
    kpi_h = Inches(2.4)
    grid_coords = [
        (Inches(6.9), Inches(1.7)),  # يمين أعلى
        (Inches(0.8), Inches(1.7)),  # يسار أعلى
        (Inches(6.9), Inches(4.5)),  # يمين أسفل
        (Inches(0.8), Inches(4.5))   # يسار أسفل
    ]

    for idx, (val, desc) in enumerate(mod["s4_kpis"]):
        gx, gy = grid_coords[idx]
        kpi_card = s4.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, gx, gy, kpi_w, kpi_h)
        kpi_card.fill.solid()
        kpi_card.fill.fore_color.rgb = COLOR_SLATE_BOX
        kpi_card.line.color.rgb = accent
        kpi_card.line.width = Pt(1)

        ktf = kpi_card.text_frame
        ktf.word_wrap = True
        ktf.margin_right = Inches(0.4)
        ktf.margin_top = Inches(0.3)
        ktf.margin_left = Inches(0.4)

        kp0 = ktf.paragraphs[0]
        kp0.text = val
        kp0.font.size = Pt(36)
        kp0.font.bold = True
        kp0.font.color.rgb = COLOR_SAND_GOLD
        kp0.alignment = PP_ALIGN.RIGHT

        kp1 = ktf.add_paragraph()
        kp1.text = desc
        kp1.font.size = Pt(13)
        kp1.font.color.rgb = COLOR_WHITE
        kp1.alignment = PP_ALIGN.RIGHT

    # إضافة ملاحظات المتحدث للملف
    for s in [s1, s2, s3, s4]:
        ntf = s.notes_slide.notes_text_frame
        ntf.text = f"إرشادات المتحدث الرسمية أمام الحضور:\n{mod['notes']}"

    prs.save(mod["filename"])
    print(f"✅ تم إنشاء: {mod['filename']}")
    return mod["filename"]

# توليد الملفات الأربعة وتحميلها
out_files = []
for m in MODULES:
    f_name = generate_professional_pptx(m)
    out_files.append(f_name)

print("\n🎉 تم بنجاح بناء العروض الأربعة وفق أعلى مقاييس التصميم المؤسسي!")
for f in out_files:
    files.download(f)

🏛️ جاري بناء حزم العروض التقديمية الاحترافية لهيئة الشارقة للآثار...
✅ تم إنشاء: العرض_1_مقدمة_الذكاء_الاصطناعي_الآثار.pptx
✅ تم إنشاء: العرض_2_توثيق_المواقع_والتسجيل_الميداني.pptx
✅ تم إنشاء: العرض_3_الكشف_والتنبؤ_والاستشعار_عن_بعد.pptx
✅ تم إنشاء: العرض_4_مراقبة_حالة_المواقع_ورصد_التدهور.pptx

🎉 تم بنجاح بناء العروض الأربعة وفق أعلى مقاييس التصميم المؤسسي!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>